# Training and evaluating a vanilla neural network

In [22]:
import json
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import numpy as np
import random

data_path = "../data"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

VANILLA_NN_MODEL_PATH = "../models/vanilla_nn/3H1L_vanilla_nn_model.pth"
SENTENCE_EMBED_TRAIN_VECTOR_PATH = "../models/vanilla_nn/3H1L_train_nn_sentence_embeddings.pkl"
SENTENCE_EMBED_VALIDATE_VECTOR_PATH = "../models/vanilla_nn/3H1L_validate_nn_sentence_embeddings.pkl"
SENTENCE_EMBED_TEST_VECTOR_PATH = "../models/vanilla_nn/3H1L_test_nn_sentence_embeddings.pkl"


### Transform data into appropriate format

Create Python list of dicts

In [23]:
def flatten_transform(target_set):

    modified_set = []

    for item in target_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_nn_ready = flatten_transform(train_nn_set)
validate_set_nn_ready = flatten_transform(validate_set)
test_set_nn_ready = flatten_transform(test_set)

Transform into dataframe and create X and y classes

In [24]:
train_nn_df = pd.DataFrame(train_set_nn_ready)
X_train_nn_text = train_nn_df["Headline"]
y_train = train_nn_df["Fake"]

validate_nn_df = pd.DataFrame(validate_set_nn_ready)
X_validate_nn_text = validate_nn_df["Headline"]
y_validate = validate_nn_df["Fake"]

test_df = pd.DataFrame(test_set_nn_ready)
X_test_nn_text = test_df["Headline"]
y_true = np.array([quiz["answer"] for quiz in test_set])

### Vanilla neural network

Get sentence embeddings

In [25]:
try:
    X_train_nn_SE = joblib.load(SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    X_validate_nn_SE = joblib.load(SENTENCE_EMBED_VALIDATE_VECTOR_PATH)
    X_test_nn_SE = joblib.load(SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Loaded model and embeddings from disk.")

except FileNotFoundError:

    print("No saved embeddings found, creating new ones...")

    # Create embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Embed both train and test sets
    X_train_nn_SE = model.encode(list(X_train_nn_text))
    X_validate_nn_SE = model.encode(list(X_validate_nn_text))
    X_test_nn_SE = model.encode(list(X_test_nn_text))

    joblib.dump(X_train_nn_SE, SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    joblib.dump(X_validate_nn_SE, SENTENCE_EMBED_VALIDATE_VECTOR_PATH)
    joblib.dump(X_test_nn_SE, SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Model and embeddings saved.")

print("Embeddings matrix shape (train):", X_train_nn_SE.shape)
print("Embeddings matrix shape (validate):", X_validate_nn_SE.shape)
print("Embeddings matrix shape (test):", X_test_nn_SE.shape)

Loaded model and embeddings from disk.
Embeddings matrix shape (train): (812, 384)
Embeddings matrix shape (validate): (268, 384)
Embeddings matrix shape (test): (268, 384)


Convert to PyTorch tensors and package into batches

In [26]:
X_train_tensor = torch.FloatTensor(X_train_nn_SE)
y_train_tensor = torch.FloatTensor(y_train.to_numpy()).unsqueeze(1)
X_validate_tensor = torch.FloatTensor(X_validate_nn_SE)
y_validate_tensor = torch.FloatTensor(y_validate.to_numpy()).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test_nn_SE)
# y_test_tensor = torch.FloatTensor(y_test.to_numpy()).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

Define neural network

In [27]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.layer3(output2)
        return output3

Train and evaluate model over 10 random seeds

In [28]:
seeds = list(range(10))
all_accuracies = []
all_probs = []

input_size = X_train_tensor.shape[1]
MAX_EPOCHS = 20

for seed in seeds:
    # Set the seed so this run starts from a known random state
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    #Rebuild the loader so its shuffling also follows this seed
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g)

    # Fresh model, optimiser and early-stopping counters for every run

    model = NeuralNetwork(input_size)
    loss_function = nn.BCEWithLogitsLoss()
    optimiser = optim.Adam(model.parameters(), lr=0.001)
    best_val_loss = float("inf")
    counter = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        losses = []
        for batch_X, batch_y in train_loader:
            optimiser.zero_grad()
            outputs = model(batch_X)
            loss = loss_function(outputs, batch_y)
            losses.append(loss.item())
            loss.backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_validate_tensor)
            val_loss = loss_function(val_outputs, y_validate_tensor)

        print(f'Epoch [{epoch+1}/{MAX_EPOCHS}], Train Loss: {(sum(losses) / len(losses)):.3f}, Val Loss: {val_loss.item():.3f}')

        current_val_loss = val_loss.item()

        if current_val_loss < best_val_loss:
            print("New best validation loss found, saving model")
            best_val_loss = current_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter == 5:
                print("Early stopping")
                break

    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        y_fake_logits = model(X_test_tensor)
        all_probs.append(torch.sigmoid(y_fake_logits).squeeze().numpy())

    probs = torch.sigmoid(y_fake_logits).squeeze().numpy()
    all_probs.append(probs)
    accuracy = (probs.reshape(-1, 4).argmax(axis=1) == y_true).mean()
    all_accuracies.append(accuracy)
    print(f"Seed {seed}: {accuracy:.2%}")

print(f"Mean: {np.mean(all_accuracies):.2%}")
print(f"Spread (std): {np.std(all_accuracies):.2%}")
print(f"Lowest: {min(all_accuracies):.2%}, Highest: {max(all_accuracies):.2%}")


Epoch [1/20], Train Loss: 0.663, Val Loss: 0.601
New best validation loss found, saving model
Epoch [2/20], Train Loss: 0.550, Val Loss: 0.543
New best validation loss found, saving model
Epoch [3/20], Train Loss: 0.504, Val Loss: 0.526
New best validation loss found, saving model
Epoch [4/20], Train Loss: 0.465, Val Loss: 0.523
New best validation loss found, saving model
Epoch [5/20], Train Loss: 0.430, Val Loss: 0.535
Epoch [6/20], Train Loss: 0.394, Val Loss: 0.533
Epoch [7/20], Train Loss: 0.352, Val Loss: 0.564
Epoch [8/20], Train Loss: 0.295, Val Loss: 0.587
Epoch [9/20], Train Loss: 0.243, Val Loss: 0.637
Early stopping
Seed 0: 46.27%
Epoch [1/20], Train Loss: 0.661, Val Loss: 0.588
New best validation loss found, saving model
Epoch [2/20], Train Loss: 0.554, Val Loss: 0.538
New best validation loss found, saving model
Epoch [3/20], Train Loss: 0.500, Val Loss: 0.522
New best validation loss found, saving model
Epoch [4/20], Train Loss: 0.467, Val Loss: 0.521
New best validatio

Save results

In [30]:
np.save("mlp_all_probs.npy", np.array(all_probs))  # keep the raw results

# Average the fake probabilities across the 10 runs
avg_probs = np.mean(all_probs, axis=0)

# Reshape headline-level averaged probabilities into 67 quizzes × 4 headlines
fake_probs_MLP = avg_probs.reshape(-1, 4)

# Predicted fake headline position (0–3)
y_pred_quiz_MLP = np.argmax(fake_probs_MLP, axis=1)

# Confidence = fake probability assigned to the selected headline
confidence_MLP = np.max(fake_probs_MLP, axis=1)

# Dates from the fixed test set
test_dates = [quiz["date"] for quiz in test_set]

# Create prediction dataframe
predictions_MLP = pd.DataFrame({
    "date": test_dates,
    "prediction": y_pred_quiz_MLP,
    "confidence": confidence_MLP
})

predictions_MLP.to_csv("../data/predictions_minilm_mlp_10seed_avg.csv", index=False)

# Accuracy of the 10-run average
y_true = np.array([quiz["answer"] for quiz in test_set])
print(f"10-seed average accuracy: {(y_pred_quiz_MLP == y_true).mean():.2%}")

10-seed average accuracy: 46.27%


Checking changes from old to new models

In [16]:
import pandas as pd

old = pd.read_csv(f"{data_path}/old/predictions_minilm_mlp.csv")
new = pd.read_csv(f"{data_path}/predictions_minilm_mlp.csv")

m = old[["date", "prediction"]].merge(
    new[["date", "prediction"]], on="date", how="outer",
    suffixes=("_old", "_new"), indicator=True
)

print(m["_merge"].value_counts())  # dates present in only one file
diff = m[(m["_merge"] == "both") & (m["prediction_old"] != m["prediction_new"])]
print(f"{len(diff)} of {len(m)} predictions changed")
print(diff.drop(columns="_merge").to_string(index=False))

_merge
both          67
left_only      0
right_only     0
Name: count, dtype: int64
5 of 67 predictions changed
      date  prediction_old  prediction_new
2020-10-03               0               3
2025-01-16               2               1
2025-06-05               1               3
2025-08-21               3               0
2026-03-05               2               0
